# Data Processing Report

Analysis of MTCNN-aligned face dataset produced by `process.py`.

Sources: CASIA-WebFace (training), LFW (held-out evaluation).
CelebA Kaggle dump lacked identity labels so was skipped — CASIA alone provides enough identities for Siamese training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

ROOT = Path('..').resolve()
df = pd.read_parquet(ROOT / 'process-data/manifest.parquet')
print(df['split'].value_counts())
print('Unique train identities:', df[df.split == 'train']['identity_id'].nunique())
df.head()

In [ ]:
counts = df[df.split.isin(['train', 'val'])].groupby('identity_id').size()
plt.figure(figsize=(8, 4))
plt.hist(counts.values, bins=50)
plt.xlabel('Images per identity')
plt.ylabel('# identities')
plt.title(f'Images per identity (median {int(counts.median())}, max {int(counts.max())})')
plt.tight_layout()
plt.show()

In [ ]:
sources = df['source'].unique()
fig, axes = plt.subplots(len(sources), 5, figsize=(12, 2.5 * len(sources)))
if len(sources) == 1:
    axes = axes.reshape(1, -1)
for row, src in enumerate(sources):
    sample = df[df.source == src].sample(min(5, len(df[df.source == src])), random_state=0)
    for col, p in enumerate(sample['path']):
        img = Image.open(ROOT / p)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_title(src, loc='left')
plt.tight_layout()
plt.show()

In [ ]:
fail_path = ROOT / 'process-data/alignment_failures.csv'
if fail_path.exists():
    fail = pd.read_csv(fail_path)
else:
    fail = pd.DataFrame()
total_attempts = len(df) + len(fail)
rate = len(fail) / total_attempts if total_attempts else 0
print(f'Aligned: {len(df)}')
print(f'Failed:  {len(fail)}')
print(f'Rate:    {rate:.2%}')

## Split summary

- **train** — used to train the embedding network (CASIA identities)
- **val** — held-out images of the same identities as train (for in-loop sanity)
- **lfw** — held-out identities for verification benchmark (different distribution)
